In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
import os
import json
import time


In [2]:
notes = pd.read_csv("../data/raw/clinical_notes.csv")

print("Original notes:", len(notes))
print(notes.columns.tolist())

Original notes: 1602
['ingest_timestamp', 'clinical_note_id', 'clean_note_text', 'creation_timestamp', 'updt_dt_tm', 'note_subject', 'note_type', 'admission_id', 'person_id']


In [3]:
# Remove broken notes
notes_clean = notes[
    notes["clean_note_text"].astype(str).str.strip() != "#NAME?"
].copy()

print("After #NAME? removal:", len(notes_clean))

After #NAME? removal: 1595


In [4]:
# Deduplicate repeated note content per patient
notes_dedup = (
    notes_clean
    .sort_values(["person_id", "creation_timestamp"])
    .drop_duplicates(
        subset=["person_id", "clean_note_text"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("After deduplication:", len(notes_dedup))
print("Patients:", notes_dedup["person_id"].nunique())

After deduplication: 1103
Patients: 50


In [5]:
print(notes_dedup.columns.tolist())

['ingest_timestamp', 'clinical_note_id', 'clean_note_text', 'creation_timestamp', 'updt_dt_tm', 'note_subject', 'note_type', 'admission_id', 'person_id']


In [6]:
# Create a new dataframe with only the relevant columns for chunking

whole_chunks = (
    notes_dedup[
        [
            "person_id",
            "creation_timestamp",
            "clean_note_text"
        ]
    ]
    .copy()
    .rename(columns={
        "clean_note_text": "chunk_text"
    })
)

# Unique identifier for every chunk/note
whole_chunks["chunk_id"] = range(len(whole_chunks))

# Ensure chronological ordering within each patient
whole_chunks = (
    whole_chunks
    .sort_values(
        ["person_id", "creation_timestamp"]
    )
    .reset_index(drop=True)
)

print("Total chunks:", len(whole_chunks))
print("Patients:", whole_chunks["person_id"].nunique())

display(whole_chunks.head())

Total chunks: 1103
Patients: 50


,person_id,creation_timestamp,chunk_text,chunk_id
0,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 08:45,"- Patient: Tomos Ellis, 15-year-old male, pre...",0
1,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 09:10,"Patient: Tomos Ellis, 15-year-old male, presen...",1
2,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 09:25,"Patient name: Tomos Ellis, 15-year-old male. N...",2
3,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:00,"Reviewed abdominal X-ray on 2026-01-07, which ...",3
4,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Patient\nTomos Ellis\n\nAge\n15\n\nSex\nMale\n...,4


In [7]:
# Load the BGE model for embeddings

bge_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

print("BGE model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BGE model loaded.


In [8]:
# Create embeddings for all chunks

chunk_texts = whole_chunks["chunk_text"].astype(str).tolist()

chunk_embeddings = bge_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding shape:", chunk_embeddings.shape)

Batches:   0%|          | 0/35 [00:00<?, ?it/s]

Embedding shape: (1103, 768)


In [9]:
RAG_QUERY = """
Retrieve the clinically relevant information needed to produce a
comprehensive longitudinal summary of this patient's clinical history,
including major diagnoses, treatments, investigations, clinical
progression, and outcomes.
""".strip()


def retrieve_patient_notes(
    person_id,
    whole_chunks,
    chunk_embeddings,
    query_text=RAG_QUERY,
    top_k=20
):
    """
    Retrieve the top-k most relevant whole-note chunks
    for one patient using BGE similarity.
    """

    # Find this patient's rows in whole_chunks
    patient_indices = np.where(
        whole_chunks["person_id"].to_numpy() == person_id
    )[0]

    # Patient-specific embeddings
    patient_embeddings = chunk_embeddings[patient_indices]

    # Encode query with the same BGE model/settings
    query_embedding = bge_model.encode(
        query_text,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    # Dot product = cosine similarity because embeddings are normalized
    scores = np.dot(
        patient_embeddings,
        query_embedding
    )

    # In case a patient has fewer than 20 notes
    k = min(top_k, len(patient_indices))

    # Top-k within THIS patient only
    local_top_indices = np.argsort(scores)[-k:][::-1]

    # Convert back to global dataframe indices
    global_top_indices = patient_indices[local_top_indices]

    # Keep metadata + similarity score
    retrieved = whole_chunks.iloc[global_top_indices].copy()

    retrieved["similarity_score"] = scores[local_top_indices]

    return retrieved

In [10]:
test_patient = whole_chunks["person_id"].iloc[0]

retrieved = retrieve_patient_notes(
    person_id=test_patient,
    whole_chunks=whole_chunks,
    chunk_embeddings=chunk_embeddings
)

print("Patient:", test_patient)
print("Retrieved notes:", len(retrieved))

display(
    retrieved[
        [
            "chunk_id",
            "person_id",
            "creation_timestamp",
            "similarity_score"
        ]
    ]
)

Patient: 028998ee-babc-4096-9b28-001bc2f9a84e
Retrieved notes: 15


,chunk_id,person_id,creation_timestamp,similarity_score
0,0,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 08:45,0.542850
8,8,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 16:00,0.542014
10,10,028998ee-babc-4096-9b28-001bc2f9a84e,08/01/2026 10:30,0.525251
4,4,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,0.523271
2,2,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 09:25,0.519711
11,11,028998ee-babc-4096-9b28-001bc2f9a84e,08/01/2026 11:15,0.517823
7,7,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 15:30,0.515500
5,5,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 11:00,0.515225
12,12,028998ee-babc-4096-9b28-001bc2f9a84e,08/01/2026 14:00,0.513148
6,6,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 13:00,0.506596


In [11]:
# Ensure chronological ordering of retrieved notes

retrieved["creation_timestamp"] = pd.to_datetime(
    retrieved["creation_timestamp"],
    dayfirst=True
)

retrieved_chronological = (
    retrieved
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

display(
    retrieved_chronological[
        [
            "chunk_id",
            "creation_timestamp",
            "similarity_score"
        ]
    ]
)

,chunk_id,creation_timestamp,similarity_score
0,0,2026-01-07 08:45:00,0.542850
1,1,2026-01-07 09:10:00,0.493814
2,2,2026-01-07 09:25:00,0.519711
3,3,2026-01-07 10:00:00,0.491517
4,4,2026-01-07 10:30:00,0.523271
5,5,2026-01-07 11:00:00,0.515225
6,6,2026-01-07 13:00:00,0.506596
7,7,2026-01-07 15:30:00,0.515500
8,8,2026-01-07 16:00:00,0.542014
9,9,2026-01-08 09:00:00,0.487196


In [12]:
# Build RAG context from retrieved notes

def build_rag_context(retrieved_chronological):
    context_parts = []

    for i, row in retrieved_chronological.iterrows():

        timestamp = row["creation_timestamp"].strftime(
            "%Y-%m-%d %H:%M"
        )

        context_parts.append(
            f"""[SOURCE NOTE {i + 1}]
Creation timestamp: {timestamp}

{row["chunk_text"]}"""
        )

    return "\n\n---\n\n".join(context_parts)


rag_context = build_rag_context(
    retrieved_chronological
)

print(rag_context[:5000])

[SOURCE NOTE 1]
Creation timestamp: 2026-01-07 08:45

 - Patient: Tomos Ellis, 15-year-old male, presented via A&E on 07/01/26 at 08:45 with severe abdominal pain over the past 3 days. - Date of birth: 2008-05-17. - NHS number: 965833270. - Triage category: 3 - Urgent. - No known allergies reported. - Current medications: None declared.- Past medical history: None documented. - Initial assessment performed by Nurse Jamie Leigh Alexander. - ED diagnosis: Constipation. - Admiting consultant: Dr. Susan Jennifer Robson. - Decision: Proceed with baseline obs and assess pain severity.
Nurse Jamie Leigh Alexander 
NMC number: 27H5222T

---

[SOURCE NOTE 2]
Creation timestamp: 2026-01-07 09:10

Patient: Tomos Ellis, 15-year-old male, presenting with abdominal pain.

- Event date/time: 07/01/26 at 09:10.
- Performed focused abdominal assessment by Nurse Jamie Leigh Alexander.
- Findings: Mild abdo distension, tenderness in lower abdo.

- NO (No guarding, rebound tenderness).
- Advise to NPO.
- 

In [13]:
import sys
from pathlib import Path

cwd = Path.cwd()
proj = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(proj))

print("Added to sys.path:", sys.path[0])

Added to sys.path: /Users/pallavi_chandanshive/projects/clinical-summarization-eval


In [14]:
from src.llm.llm import generate_rag_summary
from config.prompts import SYSTEM_PROMPT

In [15]:
# Generate RAG summary for the test patient

summary = generate_rag_summary(
    context=rag_context,
    prompt=SYSTEM_PROMPT
)

print(summary)

BadRequestError: Error code: 400 - {'error': {'message': "Unsupported value: 'temperature' does not support 0 with this model. Only the default (1) value is supported.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': 'unsupported_value'}}

In [ ]:
print("RAG context characters:", len(rag_context))

RAG context characters: 13980


In [ ]:
print("RAG context words:", len(rag_context.split()))

RAG context words: 2022


In [ ]:
patient_note_counts = (
    whole_chunks
    .groupby("person_id")
    .size()
    .sort_values(ascending=False)
)

print(patient_note_counts.head(10))

person_id
c6c45c39-cd73-49dd-818d-0a7865fe8a7f    45
136c7916-4f9b-4e5c-bf01-77e9d2c681a2    37
137b8481-4f1d-4b7f-babd-20f7117023ad    36
a9827c1c-fb54-4e5b-8bc6-d3099869e671    35
69bf7e25-abb2-4dde-857f-f1138d4d0d8a    35
6e93f9d9-213d-4f2c-a1f0-f475dacef554    34
359014a1-10e6-4bd8-9ba7-513d021c971e    34
ff8c4724-b7de-4189-bccf-cffddd4d6d44    33
04df53ea-55c1-48d9-84a1-1f15c133b29b    33
51f15281-8840-4fd0-92de-89188ab8d736    32
dtype: int64


In [ ]:
test_patient_2 = patient_note_counts[
    patient_note_counts > 20
].index[0]

print("Patient:", test_patient_2)
print("Total notes:", patient_note_counts[test_patient_2])

Patient: c6c45c39-cd73-49dd-818d-0a7865fe8a7f
Total notes: 45


In [ ]:
retrieved_2 = retrieve_patient_notes(
    person_id=test_patient_2,
    whole_chunks=whole_chunks,
    chunk_embeddings=chunk_embeddings,
    top_k=20
)

print("Patient total notes:", patient_note_counts[test_patient_2])
print("Retrieved:", len(retrieved_2))

display(
    retrieved_2[
        [
            "chunk_id",
            "person_id",
            "creation_timestamp",
            "similarity_score"
        ]
    ]
)

Patient total notes: 45
Retrieved: 20


,chunk_id,person_id,creation_timestamp,similarity_score
891,891,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,07/01/2026 17:30,0.574053
870,870,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,03/01/2026 17:30,0.566744
901,901,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,09/01/2026 17:00,0.559272
902,902,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,10/01/2026 08:00,0.559238
882,882,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,06/01/2026 09:00,0.556219
881,881,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,05/01/2026 19:00,0.551671
872,872,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,04/01/2026 08:30,0.551548
875,875,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,04/01/2026 15:30,0.550373
896,896,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,08/01/2026 18:00,0.547917
883,883,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,06/01/2026 11:30,0.547835


In [ ]:
retrieved_2["creation_timestamp"] = pd.to_datetime(
    retrieved_2["creation_timestamp"],
    dayfirst=True
)

retrieved_2_chronological = (
    retrieved_2
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

print("Before reorder:", len(retrieved_2))
print("After reorder:", len(retrieved_2_chronological))

print(
    "Same chunks:",
    set(retrieved_2["chunk_id"]) ==
    set(retrieved_2_chronological["chunk_id"])
)

display(
    retrieved_2_chronological[
        ["chunk_id", "creation_timestamp", "similarity_score"]
    ]
)

Before reorder: 20
After reorder: 20
Same chunks: True


,chunk_id,creation_timestamp,similarity_score
0,863,2026-01-03 03:15:00,0.535651
1,865,2026-01-03 04:30:00,0.538709
2,867,2026-01-03 11:00:00,0.531770
3,870,2026-01-03 17:30:00,0.566744
4,872,2026-01-04 08:30:00,0.551548
5,875,2026-01-04 15:30:00,0.550373
6,876,2026-01-04 18:00:00,0.544686
7,880,2026-01-05 16:00:00,0.531748
8,881,2026-01-05 19:00:00,0.551671
9,882,2026-01-06 09:00:00,0.556219


In [ ]:
rag_context_2 = build_rag_context(
    retrieved_2_chronological
)

print("Context notes:", len(retrieved_2_chronological))
print("Context characters:", len(rag_context_2))
print(rag_context_2[:1000])

Context notes: 20
Context characters: 19812
[SOURCE NOTE 1]
Creation timestamp: 2026-01-03 03:15

- Date: 03/01/26
 - Time: 03:15
 - Staff: Nurse Audrey Phyllis Gill
 - Chief complaint: Progressive breathlessness.
 - Past medical history: HTN, Asthma.
 - Initial management: Patient placed on low-flow oxygen therapy to improve oxygenation, with SpO2 improving from 89%to 92%.
 - ABG results reviewed: PaO2 8.5 kPa, indicating mild hypoxia; compensated respiratory alkalosis likely due to hyperventilation.
 - NT-proBNP elevated at 3,200 pg/mL, suggesting significant cardiac strain.
 - Findings escalated to the medical team for further evaluation and consideration of imaging to investigate suspected pulmonary HTN.
Nurse Audrey Phyllis Gill 
NMC number: 75Q7413G

---

[SOURCE NOTE 2]
Creation timestamp: 2026-01-03 04:30

Clerking Doctor
Dr. Sade Olowoyeye (SpR)

Presenting Complaint
Progressive breathlessness. Worsening SOB over wks. Now limits daily acti vity. No CP. No syncope. No palps. Wo

In [ ]:
summary_2 = generate_rag_summary(
    context=rag_context_2,
    prompt=SYSTEM_PROMPT
)

print(summary_2)

**Chronological Clinical Summary – 03 Jan 2026 to 10 Jan 2026**

**03 Jan 2026** – The patient (history of hypertension and asthma) presented with progressive breathlessness. Initial low‑flow nasal‑cannula oxygen raised SpO₂ from 89 % to 92 %. ABG showed mild hypoxia (PaO₂ 8.5 kPa) with compensated respiratory alkalosis. NT‑proBNP was markedly elevated (3 200 pg/mL, later 4 500 pg/mL). Examination revealed raised JVP, loud P2 and clear lungs. CXR showed cardiomegaly. A working diagnosis of chronic thromboembolic pulmonary hypertension (CTEPH) secondary to an unprovoked pulmonary embolus was made. Management instituted: enoxaparin 1.5 mg/kg daily (dose given in ED), supplemental oxygen, and plans for CT pulmonary angiogram (CTPA) and transthoracic echocardiogram. Physiotherapy introduced diaphragmatic breathing exercises.

**Later on 03 Jan** – CTPA confirmed chronic thromboembolic disease; echo pending. Anticoagulation was switched to apixaban 10 mg twice daily. Oxygen continued at low

In [ ]:
def run_rag_for_patient(
    person_id,
    whole_chunks,
    chunk_embeddings,
    query_text=RAG_QUERY,
    top_k=20
):
    # -------------------------
    # 1. Retrieve Top-K notes
    # -------------------------
    retrieved = retrieve_patient_notes(
        person_id=person_id,
        whole_chunks=whole_chunks,
        chunk_embeddings=chunk_embeddings,
        query_text=query_text,
        top_k=top_k
    )

    # -------------------------
    # 2. Convert timestamp
    # -------------------------
    retrieved = retrieved.copy()

    retrieved["creation_timestamp"] = pd.to_datetime(
        retrieved["creation_timestamp"],
        dayfirst=True
    )

    # -------------------------
    # 3. Chronological reorder
    # -------------------------
    retrieved_chronological = (
        retrieved
        .sort_values("creation_timestamp")
        .reset_index(drop=True)
    )

    # -------------------------
    # 4. Build RAG context
    # -------------------------
    rag_context = build_rag_context(
        retrieved_chronological
    )

    # -------------------------
    # 5. Generate summary
    # -------------------------
    summary = generate_rag_summary(
        context=rag_context,
        prompt=SYSTEM_PROMPT
    )

    # -------------------------
    # 6. Save provenance
    # -------------------------
    return {
        "person_id": person_id,
        "total_notes": int(
            (whole_chunks["person_id"] == person_id).sum()
        ),
        "retrieved_count": len(retrieved_chronological),
        "retrieved_chunk_ids": retrieved_chronological[
            "chunk_id"
        ].tolist(),
        "similarity_scores": retrieved_chronological[
            "similarity_score"
        ].tolist(),
        "rag_summary": summary
    }

In [ ]:
test_result = run_rag_for_patient(
    person_id=test_patient_2,
    whole_chunks=whole_chunks,
    chunk_embeddings=chunk_embeddings
)

print("Patient:", test_result["person_id"])
print("Total notes:", test_result["total_notes"])
print("Retrieved:", test_result["retrieved_count"])
print("Chunk IDs:", test_result["retrieved_chunk_ids"])

print("\nSUMMARY:\n")
print(test_result["rag_summary"])

Patient: c6c45c39-cd73-49dd-818d-0a7865fe8a7f
Total notes: 45
Retrieved: 20
Chunk IDs: [863, 865, 867, 870, 872, 875, 876, 880, 881, 882, 883, 886, 887, 891, 892, 896, 897, 900, 901, 902]

SUMMARY:

**Longitudinal Clinical Summary (03‑01‑2026 to 10‑01‑2026)**  

**03 Jan 2026** – The patient presented with progressive breathlessness. Past history included hypertension and asthma. Initial low‑flow oxygen raised SpO₂ from 89 % to 92 %; ABG showed PaO₂ 8.5 kPa with compensated respiratory alkalosis. NT‑proBNP was markedly elevated (≈3 200 pg/mL, later 4 500 pg/mL). Examination revealed raised JVP, loud P2 and clear lungs. CXR showed cardiomegaly. A provisional diagnosis of chronic thromboembolic pulmonary hypertension (CTEPH) was made. Management began with enoxaparin 1.5 mg/kg daily, supplemental O₂ (2 L NC), and plans for CT pulmonary angiogram (CTPA) and echocardiography.  

**03 Jan 2026 (later)** – CTPA confirmed chronic thromboembolic disease; echo pending. Anticoagulation was switc

In [ ]:
OUTPUT_PATH = "rag_results_checkpoint.csv"

# Resume if checkpoint already exists
if os.path.exists(OUTPUT_PATH):
    existing = pd.read_csv(OUTPUT_PATH)
    completed_ids = set(existing["person_id"].astype(str))
    rag_results = existing.to_dict("records")

    print(f"Resuming: {len(completed_ids)} patients already completed.")
else:
    completed_ids = set()
    rag_results = []

patient_ids = whole_chunks["person_id"].unique()

print("Total patients:", len(patient_ids))

for i, person_id in enumerate(patient_ids, start=1):

    if str(person_id) in completed_ids:
        print(f"{i}/{len(patient_ids)} | SKIP | {person_id}")
        continue

    print(f"{i}/{len(patient_ids)} | RUNNING | {person_id}")

    try:
        result = run_rag_for_patient(
            person_id=person_id,
            whole_chunks=whole_chunks,
            chunk_embeddings=chunk_embeddings,
            top_k=20
        )

        # Convert lists to JSON strings for safe CSV storage
        result["retrieved_chunk_ids"] = json.dumps(
            result["retrieved_chunk_ids"]
        )

        result["similarity_scores"] = json.dumps(
            result["similarity_scores"]
        )

        rag_results.append(result)

        # Checkpoint after EVERY successful patient
        pd.DataFrame(rag_results).to_csv(
            OUTPUT_PATH,
            index=False
        )

        print(
            f"    DONE | "
            f"{result['retrieved_count']}/{result['total_notes']} notes retrieved"
        )

        # Small pause between API calls
        time.sleep(1)

    except Exception as e:
        print(f"    ERROR | {person_id} | {e}")

Resuming: 18 patients already completed.
Total patients: 50
1/50 | SKIP | 028998ee-babc-4096-9b28-001bc2f9a84e
2/50 | SKIP | 04df53ea-55c1-48d9-84a1-1f15c133b29b
3/50 | RUNNING | 05192757-942f-460d-b4ff-004ec39cc5ee
    DONE | 20/23 notes retrieved
4/50 | RUNNING | 0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf
    ERROR | 0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf | Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kmvt23wjes9bf4f1wfwjc403` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199994, Requested 6730. Please try again in 48m24.767999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
5/50 | SKIP | 0f438665-d430-4adb-8acc-c3beed9e4942
6/50 | SKIP | 136c7916-4f9b-4e5c-bf01-77e9d2c681a2
7/50 | SKIP | 137b8481-4f1d-4b7f-babd-20f7117023ad
8/50 | SKIP | 1705dd0f-011a-492c-b006-b27e03f2f4ed
9/50 | SKIP | 1dbe23dc-0d1e-4

KeyboardInterrupt: 